In [1]:
# 1. Install Dependencies (Compatible with Colab T4 & Zero Drive Writes)
!pip install -q "pyarrow<24,>=19.0.0" transformers datasets accelerate peft bitsandbytes scikit-learn

import os
import re
import time
import numpy as np
import pandas as pd
import torch
from google.colab import drive
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


In [2]:
# 2. Mount Google Drive (Read-Only Access to valid.csv)
drive.mount('/content/drive')
CSV_PATH = "/content/drive/MyDrive/valid.csv"
assert os.path.exists(CSV_PATH), f"File not found at {CSV_PATH}. Check your Drive folder path!"

Mounted at /content/drive


In [3]:
# 3. Configuration (Matches PubMedBERT Split & T4 Compute Budget)
# Uses open-access identical weights so no HF login token is required
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
CONTEXT_WINDOW_WORDS = 60
MAX_LENGTH = 448        # Room for every ±60-word prompt + 32-token target (256 cut off ~1/3 of prompts)
BATCH_SIZE = 8
GRAD_ACCUM = 2          # Effective batch size = 16
EPOCHS = 3              # 3 epochs (~186 steps) prevents overfitting on 1,000 samples
LEARNING_RATE = 2e-4
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

In [4]:
# 4. Exact Same In-Memory Split as PubMedBERT Baseline
raw_df = pd.read_csv(CSV_PATH, nrows=300000).dropna(subset=["TEXT", "LOCATION", "LABEL"])
raw_df["LOCATION"] = raw_df["LOCATION"].astype(int)

def get_acronym(row):
    words = str(row["TEXT"]).split()
    loc = int(row["LOCATION"])
    return words[loc] if 0 <= loc < len(words) else ""

raw_df["ACRONYM"] = raw_df.apply(get_acronym, axis=1)
raw_df = raw_df[raw_df["ACRONYM"] != ""]

# Filter to truly ambiguous acronyms (>= 2 competing expansions)
ambig_counts = raw_df.groupby("ACRONYM")["LABEL"].nunique()
ambiguous_acronyms = ambig_counts[ambig_counts >= 2].index
ambig_df = raw_df[raw_df["ACRONYM"].isin(ambiguous_acronyms)].copy()

label_counts = ambig_df["LABEL"].value_counts()
valid_labels = label_counts[label_counts >= 30].index
ambig_df = ambig_df[ambig_df["LABEL"].isin(valid_labels)].copy()

top_ambig_acronyms = (
    ambig_df.groupby("ACRONYM")
    .filter(lambda g: g["LABEL"].nunique() >= 2)["ACRONYM"]
    .value_counts()
    .head(40)
    .index.tolist()
)

np.random.seed(SEED)
np.random.shuffle(top_ambig_acronyms)
seen_acronyms = set(top_ambig_acronyms[:30])
unseen_acronyms = set(top_ambig_acronyms[30:])

seen_pool = ambig_df[ambig_df["ACRONYM"].isin(seen_acronyms)].copy()
unseen_pool = ambig_df[ambig_df["ACRONYM"].isin(unseen_acronyms)].copy()

sampled_seen, _ = train_test_split(
    seen_pool,
    train_size=1500,
    stratify=seen_pool["LABEL"],
    random_state=SEED,
)
train_df, test_seen_df = train_test_split(
    sampled_seen,
    train_size=1000,
    test_size=500,
    stratify=sampled_seen["LABEL"],
    random_state=SEED,
)
test_unseen_df = unseen_pool.sample(n=min(500, len(unseen_pool)), random_state=SEED).copy()

print(f"Verified Exact Split -> Train: {len(train_df)} | Seen Test: {len(test_seen_df)} | Unseen Test: {len(test_unseen_df)}")

Verified Exact Split -> Train: 1000 | Seen Test: 500 | Unseen Test: 500


In [5]:
# 5. Format Prompts with Strict Exact-String Extraction Constraints
def extract_window(row, window=CONTEXT_WINDOW_WORDS):
    words = str(row["TEXT"]).split()
    loc = int(row["LOCATION"])
    if loc < 0 or loc >= len(words):
        return str(row["TEXT"]), row["ACRONYM"]
    abbr = words[loc]
    words[loc] = f"[{abbr}]"
    start = max(0, loc - window)
    end = min(len(words), loc + window + 1)
    return " ".join(words[start:end]), abbr

SYSTEM_PROMPT = (
    "You are a clinical and biomedical text parser. Expand the bracketed abbreviation "
    "based on the context. Output ONLY the exact expanded term in lowercase, with no "
    "punctuation, quotes, or explanation."
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_prompt(context_str, abbr):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Abbreviation: [{abbr}]\nContext: {context_str}\nExpansion:"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

PROMPT_BUDGET = MAX_LENGTH - 32  # Leave 32 tokens for the target expansion

def fit_prompt(row, window=CONTEXT_WINDOW_WORDS):
    # Never truncate the tokenized prompt: right-truncation cuts off the "Expansion:" line
    # and the assistant header. If a prompt is too long, narrow the word window instead.
    while True:
        context_str, abbr = extract_window(row, window)
        prompt = build_prompt(context_str, abbr)
        n_tokens = len(tokenizer(prompt, add_special_tokens=False)["input_ids"])
        if n_tokens <= PROMPT_BUDGET or window <= 5:
            return prompt, window < CONTEXT_WINDOW_WORDS
        window -= 5

def tokenize_train_row(row):
    prompt, _ = fit_prompt(row)
    target = f"{str(row['LABEL']).strip().lower()}{tokenizer.eos_token}"

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(target, truncation=True, max_length=32, add_special_tokens=False)["input_ids"]

    input_ids = prompt_ids + target_ids
    # Mask prompt tokens with -100 so loss is computed ONLY on the target expansion string
    labels = [-100] * len(prompt_ids) + target_ids
    attention_mask = [1] * len(input_ids)
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

train_records = [tokenize_train_row(row) for _, row in train_df.iterrows()]
train_ds = Dataset.from_list(train_records)

n_shrunk = sum(fit_prompt(r)[1] for d in (train_df, test_seen_df, test_unseen_df) for _, r in d.iterrows())
print(f"Prompts that needed a narrower window to fit {PROMPT_BUDGET} tokens: {n_shrunk} / {len(train_df) + len(test_seen_df) + len(test_unseen_df)}")


config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

In [6]:
# 6. Load 4-Bit Quantized Llama-3.2-1B-Instruct + Attach LoRA Adapters
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

trainable params: 5,636,096 || all params: 1,241,450,496 || trainable%: 0.4540


In [7]:
# 7. Train LoRA Adapter (All temporary checkpoints stay in Colab /content/, zero Drive writes)
training_args = TrainingArguments(
    output_dir="/content/tmp_llama_lora",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,
    fp16=True,
    logging_steps=25,
    save_strategy="no",
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, pad_to_multiple_of=8),
)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start_train = time.time()
trainer.train()
train_time = time.time() - start_train


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
25,2.953299
50,1.224762
75,0.475984
100,0.289082
125,0.246100
150,0.050211
175,0.030697


In [8]:
# 8. Fast Batched Greedy Decoding for Exact-Match Evaluation
model.eval()
tokenizer.padding_side = "left"  # Required for batched causal generation

def normalize_text(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^\w\s-]", "", s)
    return re.sub(r"\s+", " ", s).strip()

def evaluate_split(df_split, batch_size=32):
    prompts = []
    for _, row in df_split.iterrows():
        prompts.append(fit_prompt(row)[0])

    preds = []
    t0 = time.time()
    with torch.no_grad():
        for i in range(0, len(prompts), batch_size):
            batch_prompts = prompts[i : i + batch_size]
            inputs = tokenizer(
                batch_prompts,
                return_tensors="pt",
                padding=True,
                add_special_tokens=False,  # The chat template already starts with <|begin_of_text|>
            ).to(model.device)

            outputs = model.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False,        # Deterministic greedy decoding for exact string extraction
                pad_token_id=tokenizer.eos_token_id,
            )

            input_len = inputs["input_ids"].shape[1]
            for seq in outputs:
                gen_text = tokenizer.decode(seq[input_len:], skip_special_tokens=True)
                preds.append(normalize_text(gen_text))

    elapsed = time.time() - t0
    targets = [normalize_text(lbl) for lbl in df_split["LABEL"]]
    exact_match = accuracy_score(targets, preds)
    macro_f1 = f1_score(targets, preds, average="macro", zero_division=0)
    return exact_match, macro_f1, elapsed, preds, targets

seen_acc, seen_f1, seen_eval_time, seen_preds, seen_targets = evaluate_split(test_seen_df)
unseen_acc, unseen_f1, unseen_eval_time, unseen_preds, unseen_targets = evaluate_split(test_unseen_df)

peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0

print("\n" + "="*54)
print("Llama-3.2-1B-Instruct + LoRA Results (Zero Drive Writes)")
print("="*54)
print(f"GPU Used:                  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Training Wall Time:        {train_time:.2f} sec")
print(f"Inference Latency:         {(seen_eval_time/len(test_seen_df))*1000:.2f} ms/sample")
print(f"Peak GPU VRAM:             {peak_vram_mb:.1f} MB")
print("-" * 54)
print(f"Seen Ambiguous Accuracy:   {seen_acc*100:.2f}%")
print(f"Seen Macro F1:             {seen_f1*100:.2f}%")
print(f"Unseen Zero-Shot Accuracy: {unseen_acc*100:.2f}%")
print("="*54)


[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and 


Llama-3.2-1B-Instruct + LoRA Results (Zero Drive Writes)
GPU Used:                  Tesla T4
Training Wall Time:        491.32 sec
Inference Latency:         82.84 ms/sample
Peak GPU VRAM:             5057.9 MB
------------------------------------------------------
Seen Ambiguous Accuracy:   74.80%
Seen Macro F1:             60.49%
Unseen Zero-Shot Accuracy: 8.20%


In [9]:
from collections import Counter

def token_f1(pred, truth):
    pred_tokens, truth_tokens = pred.split(), truth.split()
    if not pred_tokens or not truth_tokens:
        return float(pred_tokens == truth_tokens)
    common = sum((Counter(pred_tokens) & Counter(truth_tokens)).values())
    if common == 0:
        return 0.0
    precision = common / len(pred_tokens)
    recall = common / len(truth_tokens)
    return 2 * precision * recall / (precision + recall)

seen_tok_f1 = np.mean([token_f1(p, t) for p, t in zip(seen_preds, seen_targets)])
unseen_tok_f1 = np.mean([token_f1(p, t) for p, t in zip(unseen_preds, unseen_targets)])
print(f"Seen Token-Level F1:   {seen_tok_f1*100:.2f}%")
print(f"Unseen Token-Level F1: {unseen_tok_f1*100:.2f}%")


Seen Token-Level F1:   75.67%
Unseen Token-Level F1: 14.01%


In [10]:
# Preview 5 Unseen Predictions for Qualitative Error Analysis
print("\nSample Unseen Zero-Shot Predictions:")
for i in range(5):
    print(f"  Acronym: {test_unseen_df['ACRONYM'].iloc[i]:<6} | Pred: {seen_preds[i] if False else unseen_preds[i]:<30} | True: {unseen_targets[i]}")


Sample Unseen Zero-Shot Predictions:
  Acronym: CL     | Pred: clinical                       | True: lymph
  Acronym: EC     | Pred: education                      | True: committee
  Acronym: PD     | Pred: peripheral nerve               | True: power doppler
  Acronym: CL     | Pred: clinical trial                 | True: confidence level
  Acronym: RA     | Pred: regulatory area                | True: right
